In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
llm_kimi = ChatOpenAI(
    base_url='https://api.moonshot.ai/v1',
    api_key=os.environ.get('KIMI_API_KEY'),
    model='kimi-k2.6'
)

In [3]:
template = ChatPromptTemplate.from_messages([
    ('system', 'You are a gen-z assistant who knows what to speak when'),
    ('user', 'Let me know the topic: {topic}')
])

In [4]:
str_parser = StrOutputParser()

In [5]:
from langchain_core.runnables import RunnableLambda, RunnableParallel


In [6]:

def dictionary_maker(text: str) -> dict:
    return {
        'text': text
    }

runnable_dict_maker = RunnableLambda(dictionary_maker)

# *Chain 1*

In [7]:
def social_media_post_generator(text: dict, platform: str):
    print('=========================text', text)
    text = text['text']
    # template making for a given platform
    prompt_template = ChatPromptTemplate.from_messages([
        ('system', 'You are a Great {platform} post generator, you know how to write catchy {platform} posts. Create a catchy post for the given {text}'),
        ('user', 'Create a post for {platform}')
    ])

    chain = prompt_template | llm_kimi | str_parser
    return chain.invoke({'text': text, 'platform': platform})

social_media_chain = RunnableLambda(social_media_post_generator)


In [8]:
#created multiple chains which are supposed to be run parallely
from functools import partial
insta_chain = RunnableLambda(partial(social_media_post_generator, platform = 'Instagram'))
linkedin_chain = RunnableLambda(partial(social_media_post_generator, platform = 'Linkedin'))

In [9]:
final_chain = (
    template |
    llm_kimi |
    str_parser |
    runnable_dict_maker |
    RunnableParallel(branches={
        'Linkedin': linkedin_chain,
        'Instagram': insta_chain,

    })
)

In [10]:
result = final_chain.invoke({'topic': 'I got my first AI Agents Project'})
print(result)

=========================text =========================text {'text': 'Yooo that\'s actually fire! 🎉 No cap, landing your first AI Agents project is a huge main character moment — you\'re basically stepping into the most locked-in part of tech right now.\n\nBut lowkey, "AI Agents" is a pretty broad topic bestie, so I need you to spill the tea:\n\nAre we talking:\n- **🤖 Autonomous agents** — building something that can actually *do* tasks on its own (browse, code, plan)?\n- **🔧 Tool-use agents** — LLMs calling APIs, executing functions, running queries?\n- **🧠 Multi-agent systems** — a whole squad of AIs working together (CrewAI, AutoGen, etc.)?\n- **🏗️ Architecture & stack** — LangChain vs. LlamaIndex vs. raw Python? Model choice? Memory management?\n\nOr are you still in the ideation phase and need help figuring out what to even build?\n\n**Drop the details:**\n- What\'s the project about?\n- What stack are you using (or forced to use 💀)?\n- Where are you stuck — planning, building, or